In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd

def zoom_image(image, s, method='bilinear'):  # method: 'nearest' or 'bilinear'
   
    old_h, old_w = image.shape[:2]
    new_h, new_w = int(old_h * s), int(old_w * s)
    
    if len(image.shape) == 3: # Color
        zoomed = np.zeros((new_h, new_w, image.shape[2]), dtype=np.uint8)
    else: # Grayscale
        zoomed = np.zeros((new_h, new_w), dtype=np.uint8)

    row_scale = old_h / new_h
    col_scale = old_w / new_w

    for r in range(new_h):
        for c in range(new_w):
            # Map back to original image coordinates
            rf = r * row_scale
            cf = c * col_scale

            if method == 'nearest':
                # Round to nearest integer index
                r_orig = min(int(round(rf)), old_h - 1)
                c_orig = min(int(round(cf)), old_w - 1)
                zoomed[r, c] = image[r_orig, c_orig]

            elif method == 'bilinear':
                # Get the 4 surrounding pixels
                r_low, c_low = int(rf), int(cf)
                r_high, c_high = min(r_low + 1, old_h - 1), min(c_low + 1, old_w - 1)

                # Weights
                dr, dc = rf - r_low, cf - c_low

                # Interpolation: 
                # P = P1(1-dr)(1-dc) + P2(dr)(1-dc) + P3(1-dr)(dc) + P4(dr)(dc)
                p1 = image[r_low, c_low]
                p2 = image[r_high, c_low]
                p3 = image[r_low, c_high]
                p4 = image[r_high, c_high]

                val = (p1 * (1 - dr) * (1 - dc) +
                       p2 * dr * (1 - dc) +
                       p3 * (1 - dr) * dc +
                       p4 * dr * dc)
                
                zoomed[r, c] = val.astype(np.uint8)

    return zoomed

def calculate_ssd(img1, img2):
    # Ensure dimensions match exactly for subtraction
    if img1.shape != img2.shape:
        img1 = cv2.resize(img1, (img2.shape[1], img2.shape[0]))
    diff = img1.astype(np.float32) - img2.astype(np.float32)
    
    return np.sum(diff**2) / (img1.shape[0] * img1.shape[1])


large_image__name_list = ['im01.png', 'im02.png', 'im03.png', 'taylor.jpg', 'taylor.jpg']
small_image__name_list = ['im01small.png', 'im02small.png', 'im03small.png', 'taylor_small.jpg', 'taylor_very_small.jpg']

results = []

img_path = 'images/a1images/a1q8images/'

for lg_name, sm_name in zip(large_image__name_list, small_image__name_list):
    print(img_path + lg_name)
    img_lg = cv2.imread(img_path + lg_name)
    img_sm = cv2.imread(img_path + sm_name)
    
    if img_lg is None or img_sm is None:
        print(f"Skipping: Could not find {lg_name} or {sm_name}")
        continue

    scale = img_lg.shape[0] / img_sm.shape[0]
    
    # Run algorithms
    zn_nn = zoom_image(img_sm, scale, method='nearest')
    zn_bl = zoom_image(img_sm, scale, method='bilinear')
    
    # Calculate SSD
    ssd_nn = calculate_ssd(zn_nn, img_lg)
    ssd_bl = calculate_ssd(zn_bl, img_lg)
    
    results.append({
        "Original Image": lg_name,
        "Small Image": sm_name,
        "Scale Factor": round(scale, 2),
        "SSD (Nearest)": round(ssd_nn, 4),
        "SSD (Bilinear)": round(ssd_bl, 4),
        "Improvement (%)": round(((ssd_nn - ssd_bl) / ssd_nn) * 100, 2)
    })

# Display as Table
df = pd.DataFrame(results)
display(df)

images/a1images/a1q8images/im01.png
images/a1images/a1q8images/im02.png
